# Notebook 08B - Kaggle GPU Deep Learning H=1

Controlled Kaggle UI notebook for PV ramp-rate forecasting.

Scope: `selected_physics`, `one_minute` and `one_hour`, `lookback=8`, `horizon=1`, models LSTM/GRU/BiLSTM/CNN1D/CNN-LSTM.

This notebook is intended for manual Kaggle UI execution only. It fails early if ClearML secrets or required dataset inputs are missing.

In [ ]:
from __future__ import annotations

import json
import math
import os
import random
import shutil
import subprocess
import sys
import tarfile
import time
import traceback
import zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# Notebook 08B - Kaggle GPU Deep Learning H=1 only
# ============================================================

PROJECT_NAME = "PV_ramps_forecasting_MG_VF/08_DeepLearning"
TASK_PREFIX = "NOTEBOOK_08B_selected_physics_H1_deep_learning"
DATASET_VARIANT = "selected_physics"
RESOLUTIONS = ["one_minute", "one_hour"]
LOOKBACK = 8
HORIZON = 1
TARGET_COLUMN = "ramp_rate_pu"
TIME_COLUMN = "timestamp"
MODELS = ["LSTM", "GRU", "BiLSTM", "CNN1D", "CNN-LSTM"]
RANDOM_SEED = 42
MAX_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 8
BATCH_SIZE_BY_RESOLUTION = {"one_minute": 256, "one_hour": 128}
LEARNING_RATE = 1e-3
LOSS_FUNCTION = "mse"
OPTIMIZER = "Adam"
REQUIRE_CLEARML = True

WORK_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = WORK_ROOT / "outputs" / "08_deep_learning_models"
PLOTS_DIR = OUTPUT_DIR / "plots"
HISTORY_PLOTS_DIR = OUTPUT_DIR / "training_history"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
RUN_STATE_DIR = OUTPUT_DIR / "run_state"
EXTRACT_DIR = WORK_ROOT / "input_extracted"
for directory in [OUTPUT_DIR, PLOTS_DIR, HISTORY_PLOTS_DIR, CHECKPOINT_DIR, RUN_STATE_DIR, EXTRACT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = OUTPUT_DIR / "dl_results_train_validation_test.csv"
SUMMARY_PATH = OUTPUT_DIR / "dl_results_summary.csv"
TIMING_PATH = OUTPUT_DIR / "dl_timing_report.csv"
HISTORY_PATH = OUTPUT_DIR / "dl_training_history.csv"
MODEL_COMPARISON_PATH = OUTPUT_DIR / "dl_model_comparison.csv"
BENCHMARK_PATH = OUTPUT_DIR / "dl_vs_classical_benchmark.csv"
SKIPPED_PATH = OUTPUT_DIR / "dl_skipped_failed_runs.csv"
PROMOTED_PATH = OUTPUT_DIR / "dl_promoted_candidates.csv"
FINAL_REPORT_PATH = OUTPUT_DIR / "notebook_08b_h1_kaggle_final_report.json"
ZIP_PATH = WORK_ROOT / "notebook_08b_h1_outputs.zip"

CLASSICAL_HGB_BENCHMARK = {
    ("one_minute", "selected_physics", 8, 1): {
        "validation_RMSE": 0.035209,
        "test_RMSE": 0.033882,
        "validation_skill": 0.373814,
    },
    ("one_hour", "selected_physics", 8, 1): {
        "validation_RMSE": 0.051157,
        "test_RMSE": 0.047167,
        "validation_skill": 0.377249,
    },
}

RESULT_COLUMNS = [
    "resolution", "dataset_variant", "model", "lookback", "horizon", "split",
    "MAE", "RMSE", "NRMSE", "R2", "Skill_RMSE_vs_Persistence",
    "feature_count", "sample_count",
    "training_time_seconds", "validation_time_seconds", "test_time_seconds",
    "train_evaluation_time_seconds",
    "epoch_time_mean_seconds", "epoch_time_std_seconds", "total_epochs_ran",
    "best_epoch", "early_stopping_used", "early_stopping_patience",
    "batch_size", "learning_rate", "optimizer", "loss_function",
    "backend_used", "hardware_used", "gpu_available", "gpu_count", "gpu_name",
    "status", "skip_reason", "clearml_task_id", "run_key",
]
SUMMARY_COLUMNS = [
    "status", "resolution", "dataset_variant", "model", "lookback", "horizon",
    "feature_count", "train_sample_count", "validation_sample_count", "test_sample_count",
    "train_RMSE", "validation_RMSE", "test_RMSE",
    "validation_Skill_RMSE_vs_Persistence", "test_Skill_RMSE_vs_Persistence",
    "training_time_seconds", "validation_time_seconds", "test_time_seconds",
    "epoch_time_mean_seconds", "epoch_time_std_seconds", "total_epochs_ran", "best_epoch",
    "early_stopping_used", "batch_size", "learning_rate", "optimizer", "loss_function",
    "backend_used", "hardware_used", "gpu_available", "clearml_task_id", "run_key",
]
TIMING_COLUMNS = [
    "resolution", "dataset_variant", "model", "lookback", "horizon",
    "training_time_seconds", "validation_time_seconds", "test_time_seconds",
    "train_evaluation_time_seconds", "epoch_time_mean_seconds", "epoch_time_std_seconds",
    "total_epochs_ran", "best_epoch", "early_stopping_used", "early_stopping_patience",
    "batch_size", "learning_rate", "optimizer", "loss_function", "status", "skip_reason", "run_key",
]
HISTORY_COLUMNS = [
    "resolution", "dataset_variant", "model", "lookback", "horizon",
    "epoch", "loss", "val_loss", "rmse", "val_rmse", "mae", "val_mae",
    "epoch_time_seconds", "run_key",
]
SKIPPED_COLUMNS = [
    "resolution", "dataset_variant", "model", "lookback", "horizon",
    "status", "skip_reason", "traceback", "run_key",
]


def now_stamp() -> str:
    return datetime.utcnow().strftime("%Y%m%d_%H%M%S")


RUN_STAMP = now_stamp()
TASK_NAME = f"{TASK_PREFIX}_{RUN_STAMP}"


def install_if_missing(module_name: str, pip_name: str | None = None) -> None:
    import importlib.util

    if importlib.util.find_spec(module_name) is None:
        package = pip_name or module_name
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


def preflight_backend_imports():
    install_if_missing("joblib")
    install_if_missing("sklearn", "scikit-learn")
    install_if_missing("clearml")
    install_if_missing("matplotlib")
    try:
        import tensorflow as tf
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
        import tensorflow as tf
    return tf


tf = preflight_backend_imports()
import joblib
import matplotlib.pyplot as plt
from clearml import Task
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow import keras
from tensorflow.keras import layers

os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

gpus = tf.config.list_physical_devices("GPU")
GPU_AVAILABLE = len(gpus) > 0
GPU_COUNT = len(gpus)
GPU_NAME = ";".join(getattr(gpu, "name", str(gpu)) for gpu in gpus) if gpus else ""
HARDWARE_USED = "kaggle_gpu" if GPU_AVAILABLE else "kaggle_cpu"
BACKEND_USED = "tensorflow_keras"
print(f"GPU available: {GPU_AVAILABLE} count={GPU_COUNT} names={GPU_NAME}")


def get_kaggle_secret(name: str):
    try:
        from kaggle_secrets import UserSecretsClient

        client = UserSecretsClient()
        value = client.get_secret(name)
        if value:
            return value, "kaggle_secret"
    except Exception:
        return None, "unavailable"
    return None, "missing"


def load_clearml_secrets() -> dict[str, dict[str, object]]:
    required = [
        "CLEARML_API_ACCESS_KEY",
        "CLEARML_API_SECRET_KEY",
        "CLEARML_API_HOST",
        "CLEARML_WEB_HOST",
        "CLEARML_FILES_HOST",
    ]
    statuses: dict[str, dict[str, object]] = {}
    missing: list[str] = []
    for name in required:
        value = os.environ.get(name)
        source = "environment"
        if not value:
            value, source = get_kaggle_secret(name)
            if value:
                os.environ[name] = value
        status = "SET" if value else "MISSING"
        statuses[name] = {"status": status, "source": source if value else "missing"}
        print(f"{name} {status} source={statuses[name]['source']}")
        if not value:
            missing.append(name)
    if REQUIRE_CLEARML and missing:
        write_json_atomic(OUTPUT_DIR / "clearml_preflight_failure.json", {"missing_or_empty_clearml_secrets": missing, "statuses": statuses})
        raise RuntimeError("Missing ClearML secrets; failing before training: " + ", ".join(missing))
    return statuses


def write_json_atomic(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")
    tmp.replace(path)


def write_df_atomic(path: Path, df: pd.DataFrame, columns: list[str] | None = None) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if columns is not None:
        for column in columns:
            if column not in df.columns:
                df[column] = np.nan
        df = df[columns]
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    tmp.replace(path)


def read_table(path: Path, columns: list[str]) -> pd.DataFrame:
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame(columns=columns)


results_df = read_table(RESULTS_PATH, RESULT_COLUMNS)
summary_df = read_table(SUMMARY_PATH, SUMMARY_COLUMNS)
timing_df = read_table(TIMING_PATH, TIMING_COLUMNS)
history_df = read_table(HISTORY_PATH, HISTORY_COLUMNS)
skipped_df = read_table(SKIPPED_PATH, SKIPPED_COLUMNS)


def completed_keys() -> set[str]:
    if summary_df.empty or "status" not in summary_df.columns:
        return set()
    return set(summary_df.loc[summary_df["status"].eq("success"), "run_key"].astype(str).tolist())


def outputs_roots_with_required_inputs() -> list[Path]:
    roots: list[Path] = []
    for base in [INPUT_ROOT, EXTRACT_DIR, WORK_ROOT]:
        if not base.exists():
            continue
        for path in base.rglob("outputs"):
            if not path.is_dir():
                continue
            has_required = True
            for resolution in RESOLUTIONS:
                required_relatives = [
                    f"{resolution}/notebook_04_split_windowing/{DATASET_VARIANT}/window_ready_manifest.json",
                    f"{resolution}/notebook_04_split_windowing/{DATASET_VARIANT}/train_only_standard_scaler.joblib",
                    f"{resolution}/notebook_03_physics_features/selected_features_plus_physics_informed_with_target.csv",
                ]
                for rel in required_relatives:
                    if not (path / rel).exists():
                        has_required = False
            if has_required and (path / "05_baseline_models" / "persistence_reference_by_variant_lh.csv").exists():
                roots.append(path)
    return roots


def extract_outputs_tar_if_needed() -> None:
    if outputs_roots_with_required_inputs():
        return
    for tar_path in INPUT_ROOT.rglob("outputs.tar"):
        target = EXTRACT_DIR / tar_path.parent.name
        marker = target / ".extracted"
        if marker.exists():
            continue
        target.mkdir(parents=True, exist_ok=True)
        print(f"Extracting {tar_path} to {target}")
        with tarfile.open(tar_path, "r") as tar:
            tar.extractall(target)
        marker.write_text("ok", encoding="utf-8")


def discover_outputs_roots() -> list[Path]:
    extract_outputs_tar_if_needed()
    candidates: list[Path] = []
    for base in [INPUT_ROOT, EXTRACT_DIR, WORK_ROOT]:
        if not base.exists():
            continue
        for path in base.rglob("outputs"):
            if path.is_dir() and ((path / "one_minute").exists() or (path / "one_hour").exists()):
                candidates.append(path)
    unique: list[Path] = []
    seen = set()
    for path in candidates:
        resolved = str(path.resolve())
        if resolved not in seen:
            seen.add(resolved)
            unique.append(path)
    return unique


OUTPUTS_ROOTS = discover_outputs_roots()
if not OUTPUTS_ROOTS:
    raise RuntimeError("No usable outputs/ tree found under /kaggle/input. Attach the required PV ramp Notebook 08 H1 input dataset.")
print("Discovered outputs roots:")
for root in OUTPUTS_ROOTS:
    print(root)


def find_file_in_outputs(relative_path: str) -> Path:
    rel = Path(relative_path)
    for outputs_root in OUTPUTS_ROOTS:
        candidate = outputs_root / rel
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


def remap_manifest_path(value: str | Path, outputs_root: Path, fallback_relative: str) -> Path:
    raw = str(value).replace("\\", "/")
    marker = "/outputs/"
    if marker in raw:
        rel = raw.split(marker, 1)[1]
        candidate = outputs_root / rel
        if candidate.exists():
            return candidate
    candidate = outputs_root / fallback_relative
    if candidate.exists():
        return candidate
    for root in OUTPUTS_ROOTS:
        candidate = root / fallback_relative
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not remap manifest path {value}; expected {fallback_relative}")


def load_manifest(resolution: str) -> tuple[Path, dict, Path]:
    rel = f"{resolution}/notebook_04_split_windowing/{DATASET_VARIANT}/window_ready_manifest.json"
    manifest_path = find_file_in_outputs(rel)
    outputs_root = manifest_path.parents[3]
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    return manifest_path, manifest, outputs_root


def validate_required_inputs() -> dict[str, dict[str, object]]:
    inventory: dict[str, dict[str, object]] = {}
    missing: list[str] = []
    for resolution in RESOLUTIONS:
        try:
            manifest_path, manifest, outputs_root = load_manifest(resolution)
            source = remap_manifest_path(
                manifest["source_path"],
                outputs_root,
                f"{resolution}/notebook_03_physics_features/selected_features_plus_physics_informed_with_target.csv",
            )
            scaler = remap_manifest_path(
                manifest["scaler_file"],
                outputs_root,
                f"{resolution}/notebook_04_split_windowing/{DATASET_VARIANT}/train_only_standard_scaler.joblib",
            )
            persistence = find_file_in_outputs("05_baseline_models/persistence_reference_by_variant_lh.csv")
            inventory[resolution] = {
                "manifest_path": str(manifest_path),
                "source_path": str(source),
                "scaler_path": str(scaler),
                "persistence_reference_path": str(persistence),
                "row_count": int(manifest.get("row_count", 0)),
                "feature_count": int(manifest.get("feature_count", 0)),
                "split_rows": {name: int(meta["row_count"]) for name, meta in manifest["split_summary"].items()},
            }
        except Exception as exc:
            missing.append(f"{resolution}: {type(exc).__name__}: {exc}")
    write_json_atomic(OUTPUT_DIR / "input_inventory_kaggle.json", {"inventory": inventory, "missing": missing})
    if missing:
        raise RuntimeError("Required inputs missing; failing before training: " + json.dumps(missing, indent=2))
    return inventory


clearml_secret_statuses = load_clearml_secrets()
input_inventory = validate_required_inputs()

task = Task.init(
    project_name=PROJECT_NAME,
    task_name=TASK_NAME,
    reuse_last_task_id=False,
    auto_connect_arg_parser=False,
    auto_connect_frameworks=False,
    auto_connect_streams=False,
)
task.connect(
    {
        "dataset_variant": DATASET_VARIANT,
        "resolutions": RESOLUTIONS,
        "lookback": LOOKBACK,
        "horizon": HORIZON,
        "models": MODELS,
        "epochs_max": MAX_EPOCHS,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "optimizer": OPTIMIZER,
        "learning_rate": LEARNING_RATE,
        "loss": LOSS_FUNCTION,
        "no_sample_caps_for_official_results": True,
    },
    name="notebook_08b_h1_config",
)
TASK_ID = task.id
try:
    TASK_URL = task.get_output_log_web_page()
except Exception:
    TASK_URL = ""
print(f"ClearML task created: {TASK_ID}")


def load_persistence_refs() -> pd.DataFrame:
    path = find_file_in_outputs("05_baseline_models/persistence_reference_by_variant_lh.csv")
    return pd.read_csv(path)


persistence_refs = load_persistence_refs()


def persistence_rmse(resolution: str, split_name: str) -> float:
    mask = (
        persistence_refs["resolution"].astype(str).eq(resolution)
        & persistence_refs["dataset_variant"].astype(str).eq(DATASET_VARIANT)
        & persistence_refs["lookback"].astype(int).eq(LOOKBACK)
        & persistence_refs["horizon"].astype(int).eq(HORIZON)
        & persistence_refs["split_name"].astype(str).eq(split_name)
    )
    if not bool(mask.any()):
        raise RuntimeError(f"Missing persistence RMSE reference for {resolution}/{split_name}/L{LOOKBACK}/H{HORIZON}")
    return float(persistence_refs.loc[mask, "persistence_RMSE"].iloc[0])


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, p_rmse: float) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1)
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(math.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred)) if len(y_true) > 1 else float("nan")
    return {
        "MAE": mae,
        "RMSE": rmse,
        "NRMSE": rmse / 2.0,
        "R2": r2,
        "Skill_RMSE_vs_Persistence": float(1.0 - rmse / p_rmse) if p_rmse > 0 else float("nan"),
    }


def build_windows(features: np.ndarray, target: np.ndarray, lookback: int, horizon: int) -> tuple[np.ndarray, np.ndarray]:
    count = max(0, len(target) - lookback - horizon + 1)
    if count <= 0:
        return np.empty((0, lookback, features.shape[1]), dtype=np.float32), np.empty((0,), dtype=np.float32)
    view = np.lib.stride_tricks.sliding_window_view(features, window_shape=lookback, axis=0)
    x = np.moveaxis(view[:count], -1, 1)
    x = np.ascontiguousarray(x, dtype=np.float32)
    y_start = lookback + horizon - 1
    y = np.ascontiguousarray(target[y_start : y_start + count], dtype=np.float32)
    return x, y


def load_resolution_data(resolution: str) -> dict[str, object]:
    manifest_path, manifest, outputs_root = load_manifest(resolution)
    source_path = remap_manifest_path(
        manifest["source_path"],
        outputs_root,
        f"{resolution}/notebook_03_physics_features/selected_features_plus_physics_informed_with_target.csv",
    )
    scaler_path = remap_manifest_path(
        manifest["scaler_file"],
        outputs_root,
        f"{resolution}/notebook_04_split_windowing/{DATASET_VARIANT}/train_only_standard_scaler.joblib",
    )
    feature_columns = list(manifest["feature_columns"])
    if TARGET_COLUMN in feature_columns or TIME_COLUMN in feature_columns:
        raise RuntimeError(f"Leakage-risk feature columns in manifest for {resolution}.")
    usecols = [TIME_COLUMN] + feature_columns + [TARGET_COLUMN]
    df = pd.read_csv(source_path, usecols=usecols)
    df[TIME_COLUMN] = pd.to_datetime(df[TIME_COLUMN], errors="coerce")
    if int(df[TIME_COLUMN].isna().sum()) > 0:
        raise RuntimeError(f"Invalid timestamps in {source_path}")
    if not df[TIME_COLUMN].is_monotonic_increasing:
        raise RuntimeError(f"Timestamps are not sorted in {source_path}")
    if df.columns[-1] != TARGET_COLUMN:
        raise RuntimeError(f"Target is not last column in {source_path}")
    scaler = joblib.load(scaler_path)
    splits: dict[str, dict[str, np.ndarray]] = {}
    for split_name, meta in manifest["split_summary"].items():
        start = int(meta["start_row_in_dataset"])
        end = int(meta["end_row_exclusive_in_dataset"])
        split = df.iloc[start:end]
        raw_x = split[feature_columns].to_numpy(dtype=np.float64, copy=True)
        scaled_x = scaler.transform(raw_x).astype(np.float32, copy=False)
        y = pd.to_numeric(split[TARGET_COLUMN], errors="coerce").to_numpy(dtype=np.float32)
        if not np.isfinite(scaled_x).all() or not np.isfinite(y).all():
            raise RuntimeError(f"Non-finite values found after scaling for {resolution}/{split_name}.")
        xw, yw = build_windows(scaled_x, y, LOOKBACK, HORIZON)
        splits[split_name] = {"X": xw, "y": yw}
    del df
    return {"feature_columns": feature_columns, "feature_count": len(feature_columns), "splits": splits}


def build_model(model_name: str, lookback: int, feature_count: int) -> keras.Model:
    inputs = keras.Input(shape=(lookback, feature_count), name="input_window")
    if model_name == "LSTM":
        x = layers.LSTM(32, name="lstm_32")(inputs)
        x = layers.Dropout(0.20)(x)
        x = layers.Dense(16, activation="relu")(x)
    elif model_name == "GRU":
        x = layers.GRU(32, name="gru_32")(inputs)
        x = layers.Dropout(0.20)(x)
        x = layers.Dense(16, activation="relu")(x)
    elif model_name == "BiLSTM":
        x = layers.Bidirectional(layers.LSTM(24), name="bilstm_24")(inputs)
        x = layers.Dropout(0.20)(x)
        x = layers.Dense(16, activation="relu")(x)
    elif model_name == "CNN1D":
        x = layers.Conv1D(32, kernel_size=3, padding="causal", activation="relu", name="conv1d_32")(inputs)
        x = layers.GlobalAveragePooling1D()(x)
        x = layers.Dense(16, activation="relu")(x)
    elif model_name == "CNN-LSTM":
        x = layers.Conv1D(32, kernel_size=3, padding="causal", activation="relu", name="conv1d_32")(inputs)
        x = layers.LSTM(32, name="lstm_after_conv")(x)
        x = layers.Dropout(0.20)(x)
    else:
        raise ValueError(model_name)
    outputs = layers.Dense(1, name="ramp_rate_pu_hat")(x)
    model = keras.Model(inputs, outputs, name=model_name.replace("-", "_").lower())
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=LOSS_FUNCTION,
        metrics=[keras.metrics.RootMeanSquaredError(name="rmse"), keras.metrics.MeanAbsoluteError(name="mae")],
    )
    return model


class EpochTimer(keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.epoch_times: list[float] = []
        self._start = 0.0

    def on_epoch_begin(self, epoch, logs=None):
        self._start = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        self.epoch_times.append(float(time.perf_counter() - self._start))


def safe_name(text: str) -> str:
    return text.replace("/", "_").replace("-", "_").replace(" ", "_")


def run_key(resolution: str, model_name: str) -> str:
    return f"{resolution}__{DATASET_VARIANT}__{safe_name(model_name)}__L{LOOKBACK}__H{HORIZON}"


def plot_history(history_rows: pd.DataFrame, key: str, title: str) -> None:
    if history_rows.empty:
        return
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history_rows["epoch"], history_rows["loss"], label="train_loss")
    axes[0].plot(history_rows["epoch"], history_rows["val_loss"], label="validation_loss")
    axes[0].set_title(f"{title} - loss")
    axes[0].set_xlabel("epoch")
    axes[0].legend()
    axes[0].grid(alpha=0.25)
    axes[1].plot(history_rows["epoch"], history_rows["rmse"], label="train_rmse")
    axes[1].plot(history_rows["epoch"], history_rows["val_rmse"], label="validation_rmse")
    axes[1].set_title(f"{title} - RMSE")
    axes[1].set_xlabel("epoch")
    axes[1].legend()
    axes[1].grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(HISTORY_PLOTS_DIR / f"{key}_training_curves.png", dpi=150)
    plt.close(fig)


def plot_predictions(y_true: np.ndarray, y_pred: np.ndarray, key: str, split_name: str, title: str) -> None:
    n = min(1500, len(y_true))
    idx = np.linspace(0, len(y_true) - 1, n, dtype=int) if len(y_true) > n else np.arange(len(y_true))
    yt = y_true[idx]
    yp = y_pred.reshape(-1)[idx]
    residual = yt - yp
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(yt, label="actual", linewidth=1)
    axes[0].plot(yp, label="predicted", linewidth=1)
    axes[0].set_title(f"{title} - predicted vs actual {split_name}")
    axes[0].legend()
    axes[0].grid(alpha=0.25)
    axes[1].plot(residual, linewidth=1)
    axes[1].set_title(f"{title} - residuals over sampled time {split_name}")
    axes[1].grid(alpha=0.25)
    axes[2].hist(residual, bins=60)
    axes[2].set_title(f"{title} - residual distribution {split_name}")
    axes[2].grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / f"{key}_{split_name}_prediction_residuals.png", dpi=150)
    plt.close(fig)


def save_all_tables() -> None:
    write_df_atomic(RESULTS_PATH, results_df, RESULT_COLUMNS)
    write_df_atomic(SUMMARY_PATH, summary_df, SUMMARY_COLUMNS)
    write_df_atomic(TIMING_PATH, timing_df, TIMING_COLUMNS)
    write_df_atomic(HISTORY_PATH, history_df, HISTORY_COLUMNS)
    write_df_atomic(SKIPPED_PATH, skipped_df, SKIPPED_COLUMNS)
    update_comparison_tables()


def update_comparison_tables() -> None:
    if summary_df.empty:
        write_df_atomic(MODEL_COMPARISON_PATH, pd.DataFrame(columns=[
            "resolution", "dataset_variant", "model", "lookback", "horizon",
            "validation_RMSE", "test_RMSE", "validation_Skill_RMSE_vs_Persistence",
            "training_time_seconds", "total_epochs_ran", "best_epoch", "status",
        ]))
        write_df_atomic(BENCHMARK_PATH, pd.DataFrame())
        write_df_atomic(PROMOTED_PATH, pd.DataFrame())
        return
    success = summary_df[summary_df["status"].eq("success")].copy()
    comparison = success.sort_values(["validation_RMSE", "test_RMSE"]).copy()
    write_df_atomic(MODEL_COMPARISON_PATH, comparison)

    bench_rows = []
    for _, row in success.iterrows():
        key = (row["resolution"], row["dataset_variant"], int(row["lookback"]), int(row["horizon"]))
        bench = CLASSICAL_HGB_BENCHMARK.get(key, {})
        classical_val = bench.get("validation_RMSE", np.nan)
        bench_rows.append({
            "resolution": row["resolution"],
            "dataset_variant": row["dataset_variant"],
            "model": row["model"],
            "lookback": int(row["lookback"]),
            "horizon": int(row["horizon"]),
            "dl_validation_RMSE": row["validation_RMSE"],
            "dl_test_RMSE": row["test_RMSE"],
            "classical_benchmark_model": "HistGradientBoosting",
            "classical_validation_RMSE": classical_val,
            "classical_test_RMSE": bench.get("test_RMSE", np.nan),
            "dl_beats_classical_validation_RMSE": bool(row["validation_RMSE"] < classical_val) if np.isfinite(classical_val) else False,
            "validation_RMSE_delta_vs_classical": float(row["validation_RMSE"] - classical_val) if np.isfinite(classical_val) else np.nan,
            "status": row["status"],
        })
    bench_df = pd.DataFrame(bench_rows)
    write_df_atomic(BENCHMARK_PATH, bench_df)
    promoted_rows = []
    if not comparison.empty:
        best = comparison.iloc[0]
        promoted_rows.append({
            "candidate_category": "best_dl_h1_validation_rmse",
            "resolution": best["resolution"],
            "dataset_variant": best["dataset_variant"],
            "model": best["model"],
            "lookback": int(best["lookback"]),
            "horizon": int(best["horizon"]),
            "validation_RMSE": best["validation_RMSE"],
            "test_RMSE": best["test_RMSE"],
            "rationale": "Lowest validation RMSE among completed Notebook 08B H=1 DL runs.",
            "status": "candidate_pending_review",
        })
    write_df_atomic(PROMOTED_PATH, pd.DataFrame(promoted_rows))


def plot_model_comparisons() -> None:
    success = summary_df[summary_df["status"].eq("success")].copy()
    if success.empty:
        return
    for metric, filename, ylabel in [
        ("validation_RMSE", "model_comparison_validation_rmse.png", "Validation RMSE"),
        ("test_RMSE", "model_comparison_test_rmse.png", "Test RMSE"),
        ("validation_Skill_RMSE_vs_Persistence", "model_comparison_skill_vs_persistence.png", "Skill RMSE vs Persistence"),
        ("training_time_seconds", "training_time_vs_validation_rmse.png", "Training time seconds"),
    ]:
        fig, ax = plt.subplots(figsize=(12, 5))
        labels = success["resolution"].astype(str) + " / " + success["model"].astype(str)
        if filename == "training_time_vs_validation_rmse.png":
            ax.scatter(success["training_time_seconds"], success["validation_RMSE"])
            for label, x, y in zip(labels, success["training_time_seconds"], success["validation_RMSE"]):
                ax.annotate(label, (x, y), fontsize=8, rotation=20)
            ax.set_xlabel("Training time seconds")
            ax.set_ylabel("Validation RMSE")
        else:
            ax.bar(labels, success[metric])
            ax.set_ylabel(ylabel)
            ax.tick_params(axis="x", rotation=45, labelsize=8)
        ax.set_title(filename.replace("_", " ").replace(".png", ""))
        ax.grid(alpha=0.25)
        fig.tight_layout()
        fig.savefig(PLOTS_DIR / filename, dpi=150)
        plt.close(fig)

    fig, ax = plt.subplots(figsize=(12, 5))
    labels = success["resolution"].astype(str) + " / " + success["model"].astype(str)
    ax.bar(labels, success["epoch_time_mean_seconds"])
    ax.set_ylabel("Mean epoch time seconds")
    ax.set_title("Epoch time comparison")
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "epoch_time_comparison.png", dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(labels, success["best_epoch"])
    ax.set_ylabel("Best epoch")
    ax.set_title("Best epoch comparison")
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "best_epoch_comparison.png", dpi=150)
    plt.close(fig)


def append_skipped(resolution: str, model_name: str, status: str, reason: str, tb: str, key: str) -> None:
    global skipped_df
    row = {
        "resolution": resolution,
        "dataset_variant": DATASET_VARIANT,
        "model": model_name,
        "lookback": LOOKBACK,
        "horizon": HORIZON,
        "status": status,
        "skip_reason": reason,
        "traceback": tb,
        "run_key": key,
    }
    skipped_df = pd.concat([skipped_df, pd.DataFrame([row])], ignore_index=True)
    write_df_atomic(SKIPPED_PATH, skipped_df, SKIPPED_COLUMNS)


def train_one_model(resolution: str, data: dict[str, object], model_name: str) -> None:
    global results_df, summary_df, timing_df, history_df
    key = run_key(resolution, model_name)
    if key in completed_keys():
        print(f"Skipping completed run: {key}")
        return
    run_state_path = RUN_STATE_DIR / f"{key}.json"
    write_json_atomic(run_state_path, {"status": "started", "run_key": key, "started_at": now_stamp()})
    splits = data["splits"]
    feature_count = int(data["feature_count"])
    X_train, y_train = splits["train"]["X"], splits["train"]["y"]
    X_val, y_val = splits["validation"]["X"], splits["validation"]["y"]
    X_test, y_test = splits["test"]["X"], splits["test"]["y"]
    batch_size = BATCH_SIZE_BY_RESOLUTION[resolution]
    model = build_model(model_name, LOOKBACK, feature_count)
    checkpoint_path = CHECKPOINT_DIR / f"{key}.keras"
    if checkpoint_path.exists():
        try:
            model = keras.models.load_model(checkpoint_path)
            print(f"Loaded checkpoint for {key}")
        except Exception as exc:
            print(f"Checkpoint load failed for {key}: {exc}; training fresh model.")
    timer = EpochTimer()
    callbacks = [
        timer,
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=str(checkpoint_path),
            monitor="val_loss",
            save_best_only=True,
            verbose=0,
        ),
    ]
    started = time.perf_counter()
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=MAX_EPOCHS,
        batch_size=batch_size,
        shuffle=False,
        callbacks=callbacks,
        verbose=2,
    )
    training_time = float(time.perf_counter() - started)
    epochs_ran = len(history.history.get("loss", []))
    val_losses = history.history.get("val_loss", [])
    best_epoch = int(np.argmin(val_losses) + 1) if val_losses else epochs_ran
    epoch_mean = float(np.mean(timer.epoch_times)) if timer.epoch_times else float("nan")
    epoch_std = float(np.std(timer.epoch_times)) if timer.epoch_times else float("nan")

    def predict_and_score(split_name: str, X: np.ndarray, y: np.ndarray) -> tuple[dict[str, float], np.ndarray, float]:
        start = time.perf_counter()
        pred = model.predict(X, batch_size=batch_size, verbose=0).reshape(-1)
        metrics = compute_metrics(y, pred, persistence_rmse(resolution, split_name))
        elapsed = float(time.perf_counter() - start)
        return metrics, pred, elapsed

    train_metrics, train_pred, train_eval_time = predict_and_score("train", X_train, y_train)
    val_metrics, val_pred, validation_time = predict_and_score("validation", X_val, y_val)
    test_metrics, test_pred, test_time = predict_and_score("test", X_test, y_test)

    rows = []
    for split_name, metrics, sample_count in [
        ("train", train_metrics, len(y_train)),
        ("validation", val_metrics, len(y_val)),
        ("test", test_metrics, len(y_test)),
    ]:
        rows.append({
            "resolution": resolution,
            "dataset_variant": DATASET_VARIANT,
            "model": model_name,
            "lookback": LOOKBACK,
            "horizon": HORIZON,
            "split": split_name,
            **metrics,
            "feature_count": feature_count,
            "sample_count": int(sample_count),
            "training_time_seconds": training_time,
            "validation_time_seconds": validation_time,
            "test_time_seconds": test_time,
            "train_evaluation_time_seconds": train_eval_time,
            "epoch_time_mean_seconds": epoch_mean,
            "epoch_time_std_seconds": epoch_std,
            "total_epochs_ran": epochs_ran,
            "best_epoch": best_epoch,
            "early_stopping_used": True,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "batch_size": batch_size,
            "learning_rate": LEARNING_RATE,
            "optimizer": OPTIMIZER,
            "loss_function": LOSS_FUNCTION,
            "backend_used": BACKEND_USED,
            "hardware_used": HARDWARE_USED,
            "gpu_available": GPU_AVAILABLE,
            "gpu_count": GPU_COUNT,
            "gpu_name": GPU_NAME,
            "status": "success",
            "skip_reason": "",
            "clearml_task_id": TASK_ID,
            "run_key": key,
        })
    results_df = results_df[results_df["run_key"].astype(str).ne(key)] if not results_df.empty and "run_key" in results_df.columns else results_df
    results_df = pd.concat([results_df, pd.DataFrame(rows)], ignore_index=True)

    summary_row = {
        "status": "success",
        "resolution": resolution,
        "dataset_variant": DATASET_VARIANT,
        "model": model_name,
        "lookback": LOOKBACK,
        "horizon": HORIZON,
        "feature_count": feature_count,
        "train_sample_count": len(y_train),
        "validation_sample_count": len(y_val),
        "test_sample_count": len(y_test),
        "train_RMSE": train_metrics["RMSE"],
        "validation_RMSE": val_metrics["RMSE"],
        "test_RMSE": test_metrics["RMSE"],
        "validation_Skill_RMSE_vs_Persistence": val_metrics["Skill_RMSE_vs_Persistence"],
        "test_Skill_RMSE_vs_Persistence": test_metrics["Skill_RMSE_vs_Persistence"],
        "training_time_seconds": training_time,
        "validation_time_seconds": validation_time,
        "test_time_seconds": test_time,
        "epoch_time_mean_seconds": epoch_mean,
        "epoch_time_std_seconds": epoch_std,
        "total_epochs_ran": epochs_ran,
        "best_epoch": best_epoch,
        "early_stopping_used": True,
        "batch_size": batch_size,
        "learning_rate": LEARNING_RATE,
        "optimizer": OPTIMIZER,
        "loss_function": LOSS_FUNCTION,
        "backend_used": BACKEND_USED,
        "hardware_used": HARDWARE_USED,
        "gpu_available": GPU_AVAILABLE,
        "clearml_task_id": TASK_ID,
        "run_key": key,
    }
    summary_df = summary_df[summary_df["run_key"].astype(str).ne(key)] if not summary_df.empty and "run_key" in summary_df.columns else summary_df
    summary_df = pd.concat([summary_df, pd.DataFrame([summary_row])], ignore_index=True)

    timing_row = {column: summary_row.get(column, "") for column in TIMING_COLUMNS}
    timing_row["skip_reason"] = ""
    timing_df = timing_df[timing_df["run_key"].astype(str).ne(key)] if not timing_df.empty and "run_key" in timing_df.columns else timing_df
    timing_df = pd.concat([timing_df, pd.DataFrame([timing_row])], ignore_index=True)

    hist_rows = []
    for i in range(epochs_ran):
        hist_rows.append({
            "resolution": resolution,
            "dataset_variant": DATASET_VARIANT,
            "model": model_name,
            "lookback": LOOKBACK,
            "horizon": HORIZON,
            "epoch": i + 1,
            "loss": history.history.get("loss", [np.nan] * epochs_ran)[i],
            "val_loss": history.history.get("val_loss", [np.nan] * epochs_ran)[i],
            "rmse": history.history.get("rmse", [np.nan] * epochs_ran)[i],
            "val_rmse": history.history.get("val_rmse", [np.nan] * epochs_ran)[i],
            "mae": history.history.get("mae", [np.nan] * epochs_ran)[i],
            "val_mae": history.history.get("val_mae", [np.nan] * epochs_ran)[i],
            "epoch_time_seconds": timer.epoch_times[i] if i < len(timer.epoch_times) else np.nan,
            "run_key": key,
        })
    history_df = history_df[history_df["run_key"].astype(str).ne(key)] if not history_df.empty and "run_key" in history_df.columns else history_df
    history_df = pd.concat([history_df, pd.DataFrame(hist_rows)], ignore_index=True)

    save_all_tables()
    key_history = history_df[history_df["run_key"].astype(str).eq(key)]
    plot_history(key_history, key, f"{resolution} {model_name} L={LOOKBACK} H={HORIZON}")
    plot_predictions(y_val, val_pred, key, "validation", f"{resolution} {model_name} L={LOOKBACK} H={HORIZON}")
    plot_predictions(y_test, test_pred, key, "test", f"{resolution} {model_name} L={LOOKBACK} H={HORIZON}")
    plot_model_comparisons()

    task.get_logger().report_scalar("validation_RMSE", f"{resolution}_{model_name}", iteration=LOOKBACK * 100 + HORIZON, value=float(val_metrics["RMSE"]))
    task.get_logger().report_scalar("test_RMSE", f"{resolution}_{model_name}", iteration=LOOKBACK * 100 + HORIZON, value=float(test_metrics["RMSE"]))
    write_json_atomic(run_state_path, {"status": "success", "run_key": key, "finished_at": now_stamp(), "summary": summary_row})
    print(f"Completed {key}: val_RMSE={val_metrics['RMSE']:.6f} test_RMSE={test_metrics['RMSE']:.6f}")


def upload_artifacts_to_clearml() -> None:
    artifact_paths = [
        RESULTS_PATH,
        SUMMARY_PATH,
        TIMING_PATH,
        HISTORY_PATH,
        MODEL_COMPARISON_PATH,
        BENCHMARK_PATH,
        SKIPPED_PATH,
        PROMOTED_PATH,
        FINAL_REPORT_PATH,
        ZIP_PATH,
    ]
    artifact_paths.extend(sorted(PLOTS_DIR.glob("*.png")))
    artifact_paths.extend(sorted(HISTORY_PLOTS_DIR.glob("*.png")))
    for path in artifact_paths:
        if path.exists():
            task.upload_artifact(name=path.stem[:120], artifact_object=str(path))


try:
    for resolution in RESOLUTIONS:
        print(f"Loading data for {resolution}")
        data = load_resolution_data(resolution)
        for model_name in MODELS:
            key = run_key(resolution, model_name)
            try:
                train_one_model(resolution, data, model_name)
            except Exception as exc:
                tb = traceback.format_exc()
                print(f"FAILED {key}: {type(exc).__name__}: {exc}")
                append_skipped(resolution, model_name, "failed", f"{type(exc).__name__}: {exc}", tb, key)
                save_all_tables()
        del data
        keras.backend.clear_session()

    save_all_tables()
    final_report = {
        "final_status": "completed_with_possible_model_failures" if not skipped_df.empty else "success",
        "task_name": TASK_NAME,
        "clearml_project_name": PROJECT_NAME,
        "clearml_task_id": TASK_ID,
        "clearml_task_url": TASK_URL,
        "dataset_variant": DATASET_VARIANT,
        "resolutions": RESOLUTIONS,
        "lookback": LOOKBACK,
        "horizon": HORIZON,
        "models": MODELS,
        "completed_run_count": int(len(summary_df[summary_df["status"].eq("success")])) if not summary_df.empty else 0,
        "failed_run_count": int(len(skipped_df)),
        "gpu_available": GPU_AVAILABLE,
        "gpu_count": GPU_COUNT,
        "gpu_name": GPU_NAME,
        "outputs_dir": str(OUTPUT_DIR),
        "clearml_secret_statuses": clearml_secret_statuses,
        "input_inventory": input_inventory,
    }
    write_json_atomic(FINAL_REPORT_PATH, final_report)
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()
    shutil.make_archive(str(ZIP_PATH.with_suffix("")), "zip", OUTPUT_DIR)
    upload_artifacts_to_clearml()
    task.close()
    print(json.dumps(final_report, indent=2))
    print(f"Output zip: {ZIP_PATH}")
except Exception:
    tb = traceback.format_exc()
    write_json_atomic(FINAL_REPORT_PATH, {
        "final_status": "failed",
        "error_message": tb,
        "clearml_task_id": TASK_ID if "TASK_ID" in globals() else "",
        "outputs_dir": str(OUTPUT_DIR),
    })
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()
    shutil.make_archive(str(ZIP_PATH.with_suffix("")), "zip", OUTPUT_DIR)
    try:
        upload_artifacts_to_clearml()
        task.close()
    except Exception:
        pass
    raise